In [11]:
import pandas as pd

train_df = pd.read_csv(
    r"C:\Users\ASUS\Desktop\mobile-review-sentiment-analysis\data\processed\train.csv"
)

validation_df = pd.read_csv(
    r"C:\Users\ASUS\Desktop\mobile-review-sentiment-analysis\data\processed\validation.csv"
)

test_df = pd.read_csv(
    r"C:\Users\ASUS\Desktop\mobile-review-sentiment-analysis\data\processed\test.csv"
)

In [12]:
label2id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

train_df["label"] = train_df["sentiment"].map(label2id)

validation_df["label"] = validation_df["sentiment"].map(label2id)

test_df["label"] = test_df["sentiment"].map(label2id)

In [13]:
label2id = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

id2label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

train_df["label"] = train_df["sentiment"].map(label2id)

validation_df["label"] = validation_df["sentiment"].map(label2id)

test_df["label"] = test_df["sentiment"].map(label2id)

In [14]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["clean_text", "label"]],
    preserve_index=False
)

validation_dataset = Dataset.from_pandas(
    validation_df[["clean_text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["clean_text", "label"]],
    preserve_index=False
)

In [15]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

In [16]:
def tokenize_function(examples):

    return tokenizer(
        examples["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [17]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/15988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1998 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

In [18]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/15988 [00:00<?, ? examples/s]

Map:   0%|          | 0/1998 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

In [19]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/bert_sentiment",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    report_to="none"
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [ ]:
from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

print("BERT model loaded!")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
label_map = {
    "Negative": 0,
    "Neutral": 1,
    "Positive": 2
}

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

In [ ]:
trainer.train()

In [ ]:
validation_results = trainer.evaluate(
    tokenized_validation
)

print(validation_results)

In [ ]:
test_results = trainer.evaluate(
    tokenized_test
)

print(test_results)

In [ ]:
trainer.save_model(
    "C:\\Users\\ASUS\\Desktop\\mobile-review-sentiment-analysis\\models\\bert_sentiment"
)

tokenizer.save_pretrained(
    "C:\\Users\\ASUS\\Desktop\\mobile-review-sentiment-analysis\\models\\bert_sentiment"
)

print("BERT model saved!")